In [1]:
import os
import av
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import mean_absolute_error
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BartModel, AutoImageProcessor, VideoMAEForVideoClassification
import torch.nn as nn
import torch.optim as optim

# === CONFIGURATION ===
VIDEO_PATH = "/home/spusa/Desktop/EchoNet-Dynamic/Videos"
CSV_PATH = "/home/spusa/Desktop/EchoNet-Dynamic/FileList.csv"
EMBEDDING_DIR = "Embeddings_BART"
os.makedirs(EMBEDDING_DIR, exist_ok=True)

# === Load CSV ===
df = pd.read_csv(CSV_PATH)
df = df[df["Split"].isin(["TRAIN", "VAL"])]

# === Load VideoMAE Model ===
processor = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base-finetuned-kinetics")
videomae = VideoMAEForVideoClassification.from_pretrained(
    "MCG-NJU/videomae-base-finetuned-kinetics", output_hidden_states=True
).eval()

# === Frame Sampling ===
def sample_frame_indices(clip_len, total_frames):
    end_idx = np.random.randint(clip_len, total_frames)
    start_idx = end_idx - clip_len
    return np.linspace(start_idx, end_idx - 1, clip_len).astype(np.int64)

# === Read Video Frames ===
def read_video_pyav(container, indices):
    frames = []
    container.seek(0)
    for i, frame in enumerate(container.decode(video=0)):
        if i > indices[-1]:
            break
        if i in indices:
            frames.append(frame.to_ndarray(format="rgb24"))
    return np.stack(frames)

# === Embed Using VideoMAE ===
def embed_videomae(video_path):
    try:
        container = av.open(video_path)
        total_frames = container.streams.video[0].frames
        if total_frames < 16:
            return None
        indices = sample_frame_indices(16, total_frames)
        frames = read_video_pyav(container, indices)
        inputs = processor(list(frames), return_tensors="pt")
        with torch.no_grad():
            outputs = videomae(**inputs)
        return outputs.hidden_states[-1][0, 0, :].detach().numpy()  # [768]
    except Exception as e:
        print(f"Error processing {video_path}: {e}")
        return None

# === Step 1: Generate Embeddings ===
print("🔄 Generating VideoMAE embeddings...")
for _, row in tqdm(df.iterrows(), total=len(df)):
    vid = row["FileName"]
    input_path = os.path.join(VIDEO_PATH, vid + ".avi")
    output_path = os.path.join(EMBEDDING_DIR, vid + ".npy")
    if os.path.exists(input_path) and not os.path.exists(output_path):
        emb = embed_videomae(input_path)
        if emb is not None:
            np.save(output_path, emb)

# === Step 2: Load Embeddings ===
print("📦 Loading embeddings...")
embeddings, efs = {}, {}
for _, row in df.iterrows():
    fname, ef = row["FileName"], row["EF"]
    path = os.path.join(EMBEDDING_DIR, fname + ".npy")
    if os.path.exists(path):
        embeddings[fname] = np.load(path)
        efs[fname] = ef

train_keys = df[df["Split"] == "TRAIN"]["FileName"].tolist()
test_keys = df[df["Split"] == "VAL"]["FileName"].tolist()

X_full_train = [embeddings[k] for k in train_keys if k in embeddings]
y_full_train = [efs[k] for k in train_keys if k in embeddings]
X_test = [embeddings[k] for k in test_keys if k in embeddings]
y_test = [efs[k] for k in test_keys if k in embeddings]

# === Normalize EF labels ===
max_ef = max(y_full_train + y_test)
min_ef = min(y_full_train + y_test)

def normalize(x): return (x - min_ef) / (max_ef - min_ef)
def denormalize(x): return x * (max_ef - min_ef) + min_ef

# === Step 3: BART + ReLU NN ===
class BARTRegressor(nn.Module):
    def __init__(self, bart_embed_dim=768):
        super().__init__()
        self.bart_encoder = BartModel.from_pretrained("facebook/bart-base").encoder
        print(f"📐 BART hidden size from config: {self.bart_encoder.config.d_model}")
        for param in self.bart_encoder.parameters():
            param.requires_grad = False
        self.project = nn.Linear(bart_embed_dim, bart_embed_dim)
        self.regressor = nn.Sequential(
            nn.Linear(bart_embed_dim, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        if x.shape[1] != 768:
            raise RuntimeError(f"❌ Expected input shape [B, 768], but got {x.shape}")
        x_proj = self.project(x).unsqueeze(1)  # [B, 1, 768]
        encoder_out = self.bart_encoder(inputs_embeds=x_proj)
        cls_token = encoder_out.last_hidden_state[:, 0, :]  # [B, 768]
        return self.regressor(cls_token)

# === Step 4: Predict EF (RAG + BART + ReLU) ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BARTRegressor().to(device)
optimizer = optim.Adam(model.parameters(), lr=5e-4)
criterion = nn.MSELoss()

print("🧠 Predicting EF...")
rag_predicted_efs, rag_actual_efs = [], []

for idx, test_sample in enumerate(X_test):
    print(f"\n🔍 Running Test Sample {idx+1}/{len(X_test)}")
    actual_ef = y_test[idx]
    similarities = [
        (i, cosine_similarity(test_sample.reshape(1, -1), emb.reshape(1, -1))[0][0])
        for i, emb in enumerate(X_full_train)
    ]
    top_k = sorted(similarities, key=lambda x: x[1], reverse=True)[:30]  # Increased K
    top_k_X = [X_full_train[i] for i, _ in top_k]
    top_k_y = [normalize(y_full_train[i]) for i, _ in top_k]

    X_train_tensor = torch.tensor(np.array(top_k_X), dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(top_k_y, dtype=torch.float32).view(-1, 1).to(device)
    test_tensor = torch.tensor(test_sample.reshape(1, -1), dtype=torch.float32).to(device)

    model.train()
    for _ in range(300):  # More epochs
        optimizer.zero_grad()
        pred = model(X_train_tensor)
        loss = criterion(pred, y_train_tensor)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        pred = model(test_tensor).item()
        pred_ef = denormalize(pred)

    rag_predicted_efs.append(pred_ef)
    rag_actual_efs.append(actual_ef)

# === Step 5: Evaluate ===
mae = mean_absolute_error(rag_actual_efs, rag_predicted_efs)
print(f"\n✅ Final MAE (VideoMAE + BART encoder + ReLU regressor): {mae:.2f}")


/home/spusa/.conda/envs/pytorch_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


🔄 Generating VideoMAE embeddings...


100%|██████████| 8753/8753 [00:14<00:00, 618.88it/s]


📦 Loading embeddings...
📐 BART hidden size from config: 768
🧠 Predicting EF...

🔍 Running Test Sample 1/1288

🔍 Running Test Sample 2/1288

🔍 Running Test Sample 3/1288

🔍 Running Test Sample 4/1288

🔍 Running Test Sample 5/1288

🔍 Running Test Sample 6/1288

🔍 Running Test Sample 7/1288

🔍 Running Test Sample 8/1288

🔍 Running Test Sample 9/1288

🔍 Running Test Sample 10/1288

🔍 Running Test Sample 11/1288

🔍 Running Test Sample 12/1288

🔍 Running Test Sample 13/1288

🔍 Running Test Sample 14/1288

🔍 Running Test Sample 15/1288

🔍 Running Test Sample 16/1288

🔍 Running Test Sample 17/1288

🔍 Running Test Sample 18/1288

🔍 Running Test Sample 19/1288

🔍 Running Test Sample 20/1288

🔍 Running Test Sample 21/1288

🔍 Running Test Sample 22/1288

🔍 Running Test Sample 23/1288

🔍 Running Test Sample 24/1288

🔍 Running Test Sample 25/1288

🔍 Running Test Sample 26/1288

🔍 Running Test Sample 27/1288

🔍 Running Test Sample 28/1288

🔍 Running Test Sample 29/1288

🔍 Running Test Sample 30/1288


In [2]:
import os
import av
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import mean_absolute_error
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BartModel, AutoImageProcessor, VideoMAEForVideoClassification
import torch.nn as nn
import torch.optim as optim

# === CONFIGURATION ===
VIDEO_PATH = "/home/spusa/Desktop/EchoNet-Dynamic/Videos"
CSV_PATH = "/home/spusa/Desktop/EchoNet-Dynamic/FileList.csv"
EMBEDDING_DIR = "Embeddings_BART"
os.makedirs(EMBEDDING_DIR, exist_ok=True)

# === Load CSV ===
df = pd.read_csv(CSV_PATH)
df = df[df["Split"].isin(["TRAIN", "VAL"])]

# === Load VideoMAE Model ===
processor = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base-finetuned-kinetics")
videomae = VideoMAEForVideoClassification.from_pretrained(
    "MCG-NJU/videomae-base-finetuned-kinetics", output_hidden_states=True
).eval()

# === Frame Sampling ===
def sample_frame_indices(clip_len, total_frames):
    end_idx = np.random.randint(clip_len, total_frames)
    start_idx = end_idx - clip_len
    return np.linspace(start_idx, end_idx - 1, clip_len).astype(np.int64)

# === Read Video Frames ===
def read_video_pyav(container, indices):
    frames = []
    container.seek(0)
    for i, frame in enumerate(container.decode(video=0)):
        if i > indices[-1]:
            break
        if i in indices:
            frames.append(frame.to_ndarray(format="rgb24"))
    return np.stack(frames)

# === Embed Using VideoMAE ===
def embed_videomae(video_path):
    try:
        container = av.open(video_path)
        total_frames = container.streams.video[0].frames
        if total_frames < 16:
            return None
        indices = sample_frame_indices(16, total_frames)
        frames = read_video_pyav(container, indices)
        inputs = processor(list(frames), return_tensors="pt")
        with torch.no_grad():
            outputs = videomae(**inputs)
        return outputs.hidden_states[-1][0, 0, :].detach().numpy()  # [768]
    except Exception as e:
        print(f"Error processing {video_path}: {e}")
        return None

# === Step 1: Generate Embeddings ===
print("🔄 Generating VideoMAE embeddings...")
for _, row in tqdm(df.iterrows(), total=len(df)):
    vid = row["FileName"]
    input_path = os.path.join(VIDEO_PATH, vid + ".avi")
    output_path = os.path.join(EMBEDDING_DIR, vid + ".npy")
    if os.path.exists(input_path) and not os.path.exists(output_path):
        emb = embed_videomae(input_path)
        if emb is not None:
            np.save(output_path, emb)

# === Step 2: Load Embeddings ===
print("📦 Loading embeddings...")
embeddings, efs = {}, {}
for _, row in df.iterrows():
    fname, ef = row["FileName"], row["EF"]
    path = os.path.join(EMBEDDING_DIR, fname + ".npy")
    if os.path.exists(path):
        embeddings[fname] = np.load(path)
        efs[fname] = ef

train_keys = df[df["Split"] == "TRAIN"]["FileName"].tolist()
test_keys = df[df["Split"] == "VAL"]["FileName"].tolist()

X_full_train = [embeddings[k] for k in train_keys if k in embeddings]
y_full_train = [efs[k] for k in train_keys if k in embeddings]
X_test = [embeddings[k] for k in test_keys if k in embeddings]
y_test = [efs[k] for k in test_keys if k in embeddings]

# === Normalize EF labels ===
max_ef = max(y_full_train + y_test)
min_ef = min(y_full_train + y_test)

def normalize(x): return (x - min_ef) / (max_ef - min_ef)
def denormalize(x): return x * (max_ef - min_ef) + min_ef

# === Step 3: BART + ReLU NN ===
class BARTRegressor(nn.Module):
    def __init__(self, bart_embed_dim=768):
        super().__init__()
        self.bart_encoder = BartModel.from_pretrained("facebook/bart-base").encoder
        print(f"📐 BART hidden size from config: {self.bart_encoder.config.d_model}")
        for param in self.bart_encoder.parameters():
            param.requires_grad = False
        self.project = nn.Linear(bart_embed_dim, bart_embed_dim)
        self.regressor = nn.Sequential(
            nn.Linear(bart_embed_dim, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        if x.shape[1] != 768:
            raise RuntimeError(f"❌ Expected input shape [B, 768], but got {x.shape}")
        x_proj = self.project(x).unsqueeze(1)  # [B, 1, 768]
        encoder_out = self.bart_encoder(inputs_embeds=x_proj)
        cls_token = encoder_out.last_hidden_state[:, 0, :]  # [B, 768]
        return self.regressor(cls_token)

# === Step 4: Predict EF (RAG + BART + ReLU) ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BARTRegressor().to(device)
optimizer = optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)
criterion = nn.MSELoss()

print("🧠 Predicting EF...")
rag_predicted_efs, rag_actual_efs = [], []

for idx, test_sample in enumerate(X_test):
    print(f"\n🔍 Running Test Sample {idx+1}/{len(X_test)}")
    actual_ef = y_test[idx]
    similarities = [
        (i, cosine_similarity(test_sample.reshape(1, -1), emb.reshape(1, -1))[0][0])
        for i, emb in enumerate(X_full_train)
    ]
    top_k = sorted(similarities, key=lambda x: x[1], reverse=True)[:40]  # Increased K
    top_k_X = [X_full_train[i] for i, _ in top_k]
    top_k_y = [normalize(y_full_train[i]) for i, _ in top_k]

    X_train_tensor = torch.tensor(np.array(top_k_X), dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(top_k_y, dtype=torch.float32).view(-1, 1).to(device)
    test_tensor = torch.tensor(test_sample.reshape(1, -1), dtype=torch.float32).to(device)

    model.train()
    for _ in range(400):  # More epochs
        optimizer.zero_grad()
        pred = model(X_train_tensor)
        loss = criterion(pred, y_train_tensor)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        pred = model(test_tensor).item()
        pred_ef = denormalize(pred)

    rag_predicted_efs.append(pred_ef)
    rag_actual_efs.append(actual_ef)

# === Step 5: Evaluate ===
mae = mean_absolute_error(rag_actual_efs, rag_predicted_efs)
print(f"\n✅ Final MAE (VideoMAE + BART encoder + ReLU regressor): {mae:.2f}")


🔄 Generating VideoMAE embeddings...


100%|██████████| 8753/8753 [00:13<00:00, 656.62it/s]


📦 Loading embeddings...
📐 BART hidden size from config: 768
🧠 Predicting EF...

🔍 Running Test Sample 1/1288

🔍 Running Test Sample 2/1288

🔍 Running Test Sample 3/1288

🔍 Running Test Sample 4/1288

🔍 Running Test Sample 5/1288

🔍 Running Test Sample 6/1288

🔍 Running Test Sample 7/1288

🔍 Running Test Sample 8/1288

🔍 Running Test Sample 9/1288

🔍 Running Test Sample 10/1288

🔍 Running Test Sample 11/1288

🔍 Running Test Sample 12/1288

🔍 Running Test Sample 13/1288

🔍 Running Test Sample 14/1288

🔍 Running Test Sample 15/1288

🔍 Running Test Sample 16/1288

🔍 Running Test Sample 17/1288

🔍 Running Test Sample 18/1288

🔍 Running Test Sample 19/1288

🔍 Running Test Sample 20/1288

🔍 Running Test Sample 21/1288

🔍 Running Test Sample 22/1288

🔍 Running Test Sample 23/1288

🔍 Running Test Sample 24/1288

🔍 Running Test Sample 25/1288

🔍 Running Test Sample 26/1288

🔍 Running Test Sample 27/1288

🔍 Running Test Sample 28/1288

🔍 Running Test Sample 29/1288

🔍 Running Test Sample 30/1288
